In [ ]:
import json
import stanza
import os
from pathlib import Path
import re

nlp = stanza.Pipeline('en', processors='tokenize,pos')

PROJECT_ROOT = Path("/mimer/NOBACKUP/groups/naiss2025-22-1187/coherence-driven-humans")
source_file = str(PROJECT_ROOT / "data" / "post-processing" / "cleaned_outputs.json")
target_base_dir = str(PROJECT_ROOT / "data" / "implicit_connective_data" / "disrpt_inputs")

AVAILABLE_MODELS = ['claude45', 'gpt4o', 'human', 'internvl3', 'llama4scout', 'qwen3vl']
PROMPT_TYPES = ['original', 'large']

2026-02-20 15:21:16 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-02-20 15:21:16 INFO: Downloaded file to /home/xilini/stanza_resources/resources.json
2026-02-20 15:21:16 WARNING: Language en package default expects mwt, which has been added
2026-02-20 15:21:17 INFO: Loading these models for language: en (English):
| Processor | Package         |
-------------------------------
| tokenize  | combined        |
| mwt       | combined        |
| pos       | combined_charlm |

2026-02-20 15:21:17 INFO: Using device: cuda
2026-02-20 15:21:17 INFO: Loading: tokenize
2026-02-20 15:21:17 INFO: Loading: mwt
2026-02-20 15:21:17 INFO: Loading: pos
2026-02-20 15:21:18 INFO: Done loading processors!


In [ ]:
def load_data():
    with open(source_file, 'r') as f:
        return json.load(f)


def filter_data(data, model_type, prompt_type):
    # Seed rules:
    #   original: seed 44 for claude45, seed 42 for everything else
    #   large: all three seeds (42,43,44) for humans, seed 42 only for models
    filtered = []
    for item in data:
        if item['model_type'] != model_type or item['prompt_type'] != prompt_type:
            continue
        seed = item.get('seed', 42)
        if prompt_type == 'original':
            if model_type == 'claude45':
                if seed == 44:
                    filtered.append(item)
            else:
                if seed == 42:
                    filtered.append(item)
        elif prompt_type == 'large':
            if model_type == 'human':
                filtered.append(item)
            else:
                if seed == 42:
                    filtered.append(item)
    return filtered


def split_into_segments(text):
    text = re.sub(r'\[SENT\]', '', text)
    segments = text.split('[SEP]')
    cleaned_segments = []
    for seg in segments:
        seg = re.sub(r'\s+', ' ', seg).strip()
        if seg:
            cleaned_segments.append(seg)
    return cleaned_segments

In [ ]:
def create_folder_name(model_type, prompt_type, seed=None):
    folder_parts = [model_type, prompt_type]
    # Only add seed for human + large (multiple seeds)
    if model_type == 'human' and prompt_type == 'large' and seed is not None:
        folder_parts.append(f"seed{seed}")
    return "_".join(folder_parts)


def process_segment_to_conllu(segment_text, segment_id, char_offset=0):
    doc = nlp(segment_text)
    conllu_lines = []
    token_count = 0

    for sent_idx, sentence in enumerate(doc.sentences):
        conllu_lines.append(f"# text = {sentence.text}")
        conllu_lines.append(f"# sent_id = {segment_id}.{sent_idx}")

        for token in sentence.tokens:
            for word in token.words:
                misc = f"start_char={word.start_char + char_offset}|end_char={word.end_char + char_offset}"
                line = f"{word.id}\t{word.text}\t_\t{word.upos}\t{word.xpos}\t_\t0\t_\t_\t{misc}"
                conllu_lines.append(line)
                token_count += 1
        conllu_lines.append("")

    new_char_offset = char_offset + len(segment_text) + 1
    return conllu_lines, new_char_offset, token_count


def process_story_to_conllu(segments, doc_id):
    all_lines = [f"# newdoc id = {doc_id}"]
    char_offset = 0
    token_position = 1
    segment_boundaries = []

    for seg_idx, segment_text in enumerate(segments):
        seg_start_token = token_position
        conllu_lines, char_offset, token_count = process_segment_to_conllu(
            segment_text, seg_idx, char_offset
        )
        all_lines.extend(conllu_lines)
        token_position += token_count
        segment_boundaries.append((seg_start_token, token_position - 1))

    return "\n".join(all_lines), segment_boundaries

In [ ]:
def process_model_prompt_combination(data, model_type, prompt_type):
    print(f"Processing: {model_type} - {prompt_type}")

    filtered = filter_data(data, model_type, prompt_type)
    if not filtered:
        print(f"No data found for {model_type} - {prompt_type}")
        return

    print(f"Found {len(filtered)} entries")

    seed_groups = {}
    for item in filtered:
        seed = item.get('seed', 42)
        seed_groups.setdefault(seed, []).append(item)

    print(f"Seeds found: {list(seed_groups.keys())}")

    for seed, items in seed_groups.items():
        if model_type == 'human' and prompt_type == 'large':
            folder_name = create_folder_name(model_type, prompt_type, seed)
        else:
            folder_name = create_folder_name(model_type, prompt_type)

        print(f"  Seed {seed}: {len(items)} stories -> Folder: {folder_name}")

        target_dir = Path(target_base_dir) / folder_name / "eng.pdtb.gum"
        target_dir.mkdir(parents=True, exist_ok=True)

        try:
            all_conllu = []
            all_segment_info = []

            for item in items:
                story_id = item['story_id']
                doc_id = f"story_{story_id}"
                cleaned_text = item.get('cleaned_model_output', '')
                if not cleaned_text:
                    print(f"    Warning: No cleaned_model_output for story {story_id}")
                    continue

                segments = split_into_segments(cleaned_text)
                if not segments:
                    print(f"    Warning: No segments found for story {story_id}")
                    continue

                conllu_content, segment_boundaries = process_story_to_conllu(segments, doc_id)
                all_conllu.append(conllu_content)
                all_segment_info.append({
                    'story_id': story_id,
                    'segments': segments,
                    'boundaries': segment_boundaries
                })

            with open(target_dir / "eng.pdtb.gum_test.conllu", 'w', encoding='utf-8') as f:
                f.write("\n\n".join(all_conllu))

            # Empty .rels file — populated by generate_rels_vwp_implicit.ipynb
            with open(target_dir / "eng.pdtb.gum_test.rels", 'w', encoding='utf-8') as f:
                f.write("")

            print(f"    Created: {target_dir / 'eng.pdtb.gum_test.conllu'}")
            print(f"    Created: {target_dir / 'eng.pdtb.gum_test.rels'}")

            if all_segment_info:
                sample = all_segment_info[0]
                print(f"    Sample story {sample['story_id']}: {len(sample['segments'])} segments")

        except Exception as e:
            print(f"    Error processing {model_type}/{prompt_type}: {e}")
            import traceback
            traceback.print_exc()

In [ ]:
def main(model_types=None, prompt_types=None):
    if model_types is None:
        model_types = AVAILABLE_MODELS
    elif isinstance(model_types, str):
        model_types = [model_types]

    if prompt_types is None:
        prompt_types = PROMPT_TYPES
    elif isinstance(prompt_types, str):
        prompt_types = [prompt_types]

    print(f"Processing models: {model_types}")
    print(f"Processing prompts: {prompt_types}")

    model_types = [m for m in model_types if m in AVAILABLE_MODELS]
    prompt_types = [p for p in prompt_types if p in PROMPT_TYPES]

    if not model_types or not prompt_types:
        print("No valid model/prompt types to process!")
        return

    data = load_data()
    print(f"Loaded {len(data)} total entries")

    for model_type in model_types:
        for prompt_type in prompt_types:
            try:
                process_model_prompt_combination(data, model_type, prompt_type)
            except Exception as e:
                print(f"Error processing {model_type}/{prompt_type}: {e}")

    print("Done.")

In [8]:
# Run for all models and prompts
main()

Processing models: ['claude45', 'gpt4o', 'human', 'internvl3', 'llama4scout', 'qwen3vl']
Processing prompts: ['original', 'large']

Loading data...
Loaded 1440 total entries

PROCESSING: CLAUDE45 - ORIGINAL
Found 60 entries
Seeds found: [44]

  Seed 44: 60 stories -> Folder: claude45_original
    ✓ Created: /home/xilini/disrpt25-task-OLD/data_v2/claude45_original/eng.pdtb.gum/eng.pdtb.gum_test.conllu
    ✓ Created: /home/xilini/disrpt25-task-OLD/data_v2/claude45_original/eng.pdtb.gum/eng.pdtb.gum_test.rels
    Sample story 10499: 5 segments
      Seg 0 (tokens 1-48): Al stands beside a light blue sedan in a parking lot, grippi...
      Seg 1 (tokens 49-83): Al maintains his hold on the man, who is now partially out o...
      Seg 2 (tokens 84-128): The confrontation escalates into a full physical altercation...

PROCESSING: CLAUDE45 - LARGE
Found 60 entries
Seeds found: [42]

  Seed 42: 60 stories -> Folder: claude45_large
    ✓ Created: /home/xilini/disrpt25-task-OLD/data_v2/claude45_

In [ ]:
# Or run for specific model/prompt combinations:
# main(model_types=['qwen3vl'], prompt_types=['original'])
# main(model_types='human', prompt_types='large')